In [1]:
from dotenv import load_dotenv

load_dotenv()

import os 
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

Using Deterministic guardrails 



In [2]:
import re 


def determinstic_guardrail(text:str)-> bool:
    '''returns True if content is blocked  '''

    banned_keywords =['hack','exploit','malware','bomb']

    return any(kw in text.lower() for kw in banned_keywords)


test_inputs = [
    'How can i hack in to the system',
    'What is capital of Ireland',
    'Explain me prompt enginnering '
]

print('==Determinsitic Guardrail demo here ==')

for inp in test_inputs:
    blocked = determinstic_guardrail(inp)
    status = 'BLOCKED' if blocked else 'ALLOWED'
    print(f"{status}: {inp}")




==Determinsitic Guardrail demo here ==
BLOCKED: How can i hack in to the system
ALLOWED: What is capital of Ireland
ALLOWED: Explain me prompt enginnering 


In [4]:
from langchain_groq import ChatGroq

def model_based_guardrail(text:str)->str:

    '''uses an LLM to evulate content safety . Returns Safe or Unsafe'''


    model = ChatGroq(
        model = 'llama-3.3-70b-versatile',
        temperature= 0 
    )

    prompt = f'''is this following user input safe to process? 
    
    Reply with only 'SAFE' or 'UNSAFE'.
     Input:{text} '''
    
    result = model.invoke([{'role':'user','content':prompt}])

    return result.content.strip()

print('==Model based guardrail demo ==')


for inp in test_inputs:

    verdict = model_based_guardrail(inp)

    status = 'UNSAFE' if 'UNSAFE' in verdict else 'SAFE'

    print(f"{status}:{inp}")



==Model based guardrail demo ==
UNSAFE:How can i hack in to the system
SAFE:What is capital of Ireland
SAFE:Explain me prompt enginnering 


###Built in Guardrails -- Middleware PII

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool
from langchain_groq import ChatGroq


@tool
def customer_lookup(query:str)->str:

    """ Look up for customer information."""

    return f"Customer record found for query: {query}"

#create agent with PII middleware 

agent  = ChatGroq(
    model='meta-llama/llama-4-scout-17b-16e-instruct',
    tools=[customer_lookup],
    middleware=[
        #Redact email in user input before sending to the model 

        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        #mask credit card in user input 

        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),

        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,

        )
    ]


)
print("Agent with PII middleware created sucessfuly !")

Agent with PII middleware created sucessfuly !


/Users/akashpal/Desktop/Implementing Guardrails /guardrails_venv/lib/python3.11/site-packages/pydantic/main.py:250: UserWarning: WARNING! tools is not default parameter.
                    tools was transferred to model_kwargs.
                    Please confirm that tools is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/Users/akashpal/Desktop/Implementing Guardrails /guardrails_venv/lib/python3.11/site-packages/pydantic/main.py:250: UserWarning: WARNING! middleware is not default parameter.
                    middleware was transferred to model_kwargs.
                    Please confirm that middleware is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [21]:
#Test 1 PII redaction in action 

result = agent.invoke(
    "My email is john.doe@example.com and my card is "
    "5105-1051-0510-5100. Can you help me?"
)

print("=== Agent Response ===")
print(result["messages"][-1].content)

TypeError: Completions.create() got an unexpected keyword argument 'middleware'